# 🤖 AI Interview Simulator — Colab Server

Run this notebook on **Google Colab (T4 GPU)** to launch the backend.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. Copy the ngrok URL printed at the end
4. Set `VITE_API_BASE` in your frontend to that URL

In [11]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q fastapi uvicorn[standard] python-multipart
!pip install -q sentence-transformers 'transformers>=4.40.0' torch torchaudio 'accelerate>=0.26.0'
!pip install -q 'bitsandbytes>=0.43.0'   # required for 4-bit Qwen2.5-7B quantization
!pip install -q pdfplumber docx2txt scikit-learn numpy requests httpx
!pip install -q soundfile librosa ffmpeg-python pydantic python-dotenv jiwer
!pip install -q spacy && python -m spacy download en_core_web_sm -q
!apt-get install -qq ffmpeg
print('✅ Dependencies installed')


ERROR: Operation cancelled by user
✅ Dependencies installed


In [ ]:
# ── Cell 2: GPU / VRAM check — run this BEFORE downloading models ─────────────
import torch
if torch.cuda.is_available():
    device     = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {device}')
    print(f'Total VRAM: {total_vram:.1f} GB')
    if total_vram < 12:
        print('⚠️  WARNING: Less than 12 GB VRAM detected.')
        print('   LLM scoring (Qwen2.5-7B 4-bit) may cause OOM.')
        print('   Go to Runtime → Change runtime type → T4 GPU')
    else:
        print('✅ VRAM sufficient for all models:')
        print('   Whisper large-v3-turbo  ~2.0 GB')
        print('   all-mpnet-base-v2       ~0.5 GB')
        print('   Qwen2.5-7B-Instruct 4-bit  ~4.5 GB')
        print('   ─────────────────────────────────')
        print(f'   Total used: ~7.0 GB / {total_vram:.1f} GB available')
else:
    print('❌ No GPU detected. LLM scoring will be DISABLED (falls back to cosine similarity).')
    print('   Go to Runtime → Change runtime type → T4 GPU')


In [ ]:
# ── Cell 2: Mount Google Drive (optional — for persistent storage) ────────────
MOUNT_DRIVE = False   # Set True to persist sessions across restarts
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE_PATH = '/content/drive/MyDrive/interview_simulator/storage'
else:
    STORAGE_PATH = '/content/interview_simulator/storage'

import os
os.makedirs(STORAGE_PATH, exist_ok=True)
print(f'✅ Storage at: {STORAGE_PATH}')

✅ Storage at: /content/interview_simulator/storage


In [ ]:
# ── Cell 3: Clone / upload your repo ─────────────────────────────────────────
# Option A: Clone from GitHub (replace with your repo URL)
GITHUB_REPO = 'https://github.com/FaaizBinKasim/Multimodal-AI-interview-sim.git'
!git clone {GITHUB_REPO} /content/project 2>/dev/null || git -C /content/project pull
%cd /content/project
print('✅ Project ready')

/content/project
✅ Project ready


In [ ]:
# ── Cell 4: Pre-download ML models (do this once — saves time later) ─────────
print('📥 Downloading all-mpnet-base-v2 embedding model...')
from sentence_transformers import SentenceTransformer
SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print('✅ Embedding model cached')

print('📥 Downloading Whisper large-v3-turbo ASR model...')
from transformers import pipeline
import torch
pipeline('automatic-speech-recognition', model='openai/whisper-large-v3-turbo',
         device='cuda' if torch.cuda.is_available() else 'cpu')
print('✅ Whisper model cached')
print(f'🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

📥 Downloading all-mpnet-base-v2 embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

✅ Whisper model cached
🖥️  GPU: Tesla T4


In [ ]:
# ── Cell: Environment variables ──────────────────────────────────────────────
import os

# LLM Eager Load — pre-warm Qwen2.5-7B at startup (adds ~90s to startup time)
# Set to 'false' to lazy-load on first answer (saves startup time in dev)
os.environ['LLM_EAGER_LOAD'] = 'true'

# HuggingFace token — NOT required for public models (Qwen2.5, Whisper, mpnet)
# Only needed if you use gated models. Leave empty for the default setup.
os.environ['HF_TOKEN'] = ''   # optional: paste your HF token here

print('LLM_EAGER_LOAD:', os.environ.get('LLM_EAGER_LOAD'))
print('HF_TOKEN set:',    bool(os.environ.get('HF_TOKEN')))


HF_TOKEN set: False


In [ ]:
# ── Cell 6: Install and configure ngrok ───────────────────────────────────────
!pip install -q pyngrok
from pyngrok import ngrok, conf

# Get your free authtoken at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = '3BOsAXhyUa9Q1dKSDNGEojhPVGh_3KfHVrLCH4N5cUQ9Utngz'   # paste your ngrok authtoken here
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok configured')

✅ ngrok configured                                                                                  


In [ ]:
# ── Cell: Health check + VRAM usage after startup ───────────────────────────
import requests, time, torch
time.sleep(5)  # Wait for server startup

health = requests.get(f'{public_url}/api/health', timeout=10).json()
print('Backend status:', health)

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM after startup: {used:.1f} GB / {total:.1f} GB used')
    remaining = total - used
    print(f'VRAM remaining: {remaining:.1f} GB free')
else:
    print('No GPU — LLM scoring disabled, using cosine similarity fallback')


^C
⏳ Waiting for server...
✅ Server is running!

🌐 PUBLIC BACKEND URL:
   https://unchaperoned-nonimaginarily-florentina.ngrok-free.dev

📋 Set this in your frontend:
   VITE_API_BASE=https://unchaperoned-nonimaginarily-florentina.ngrok-free.dev/api

🔗 API Docs:
   https://unchaperoned-nonimaginarily-florentina.ngrok-free.dev/docs



In [ ]:
# ── Cell 8: View server logs (run while server is active) ────────────────────
# Run this to see live server output
import sys
for line in proc.stdout:
    print(line.decode().rstrip())